# CivicGuard AI — MobileNetV2 Training (Clean Colab Notebook)

One clean run: download datasets → auto-organize into class folders → auto-detect which classes have real data → train MobileNetV2 → evaluate → save + download.

**5 target classes:** `blocked_drain`, `sewage_overflow`, `road_damage`, `fallen_tree`, `water_logging`

**Data you currently have:** `road_damage`, `fallen_tree`, `water_logging` (3 of 5). `blocked_drain` and `sewage_overflow` have no dataset yet — see the last section for how to add them later. This notebook automatically trains on whichever classes actually have images, so you don't need to manually toggle anything.

## 1. Install dependencies

In [ ]:
!pip install -q kagglehub

## 2. Download datasets

Runs entirely on Colab's own disk — nothing from your laptop needed for these.

In [ ]:
import kagglehub

road_damage_path = kagglehub.dataset_download("alvarobasily/road-damage")
pothole_path = kagglehub.dataset_download("andrewmvd/pothole-detection")
flood_path = kagglehub.dataset_download("saiharshitjami/flood-images-mask-segmentation")
tree_path = kagglehub.dataset_download("akinduhiman/urban-issues-dataset")

print("road_damage_path:", road_damage_path)
print("pothole_path:", pothole_path)
print("flood_path:", flood_path)
print("tree_path:", tree_path)

## 3. Organize into `data/processed/<class_name>/`

This builds the exact folder layout `image_dataset_from_directory` needs (one folder per class, images directly inside), pulling the right images from each dataset:

- **road_damage** ← road-damage + pothole-detection (object-detection sets — we only use the raw images, not the boxes)
- **water_logging** ← flood segmentation set (excluding mask images, keeping only real photos)
- **fallen_tree** ← the `FallenTrees/FallenTrees` folder inside the urban-issues bundle (already pure fallen-tree photos, no filtering needed)
- **blocked_drain**, **sewage_overflow** ← left empty for now (no dataset yet)

In [ ]:
import shutil
from pathlib import Path

DATA_ROOT = Path("data/processed")
CLASS_NAMES = ["blocked_drain", "sewage_overflow", "road_damage", "fallen_tree", "water_logging"]
IMG_EXTS = {".jpg", ".jpeg", ".png"}

for c in CLASS_NAMES:
    (DATA_ROOT / c).mkdir(parents=True, exist_ok=True)

def copy_images(src_dir, dest_class, prefix, exclude_keywords=None):
    """Recursively copy image files from src_dir into data/processed/dest_class,
    skipping any path containing exclude_keywords (e.g. 'mask')."""
    exclude_keywords = exclude_keywords or []
    dest = DATA_ROOT / dest_class
    count = 0
    for f in Path(src_dir).rglob("*"):
        if f.suffix.lower() not in IMG_EXTS:
            continue
        path_lower = str(f).lower()
        if any(k in path_lower for k in exclude_keywords):
            continue
        shutil.copy(f, dest / f"{prefix}_{count}{f.suffix}")
        count += 1
    print(f"Copied {count} images into '{dest_class}' from {src_dir}")

# road_damage
copy_images(road_damage_path, "road_damage", "roaddmg")
copy_images(pothole_path, "road_damage", "pothole")

# water_logging (skip mask images)
copy_images(flood_path, "water_logging", "flood", exclude_keywords=["mask"])

# fallen_tree
fallen_tree_dir = Path(tree_path) / "FallenTrees" / "FallenTrees"
copy_images(fallen_tree_dir, "fallen_tree", "urbanissues")

print()
print("Image counts per class:")
for c in CLASS_NAMES:
    n = len(list((DATA_ROOT / c).glob("*")))
    print(f"  {c}: {n}")

## 4. Auto-select classes with real data

`image_dataset_from_directory` breaks (or silently misbehaves) if a class folder is empty, so this step keeps only classes that actually have images — no manual toggling needed. Once you add data for `blocked_drain` / `sewage_overflow`, re-running this cell picks them up automatically.

In [ ]:
import shutil as _shutil

MIN_IMAGES = 1  # a class needs at least this many images to be included

ACTIVE_CLASSES = [c for c in CLASS_NAMES if len(list((DATA_ROOT / c).glob("*"))) >= MIN_IMAGES]
SKIPPED = [c for c in CLASS_NAMES if c not in ACTIVE_CLASSES]

ACTIVE_DATA_ROOT = Path("data/active")
if ACTIVE_DATA_ROOT.exists():
    _shutil.rmtree(ACTIVE_DATA_ROOT)
ACTIVE_DATA_ROOT.mkdir(parents=True)

for c in ACTIVE_CLASSES:
    _shutil.copytree(DATA_ROOT / c, ACTIVE_DATA_ROOT / c)

print("Training on:", ACTIVE_CLASSES)
if SKIPPED:
    print("Skipped (no images yet):", SKIPPED)

## 5. Config

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("artifacts")
MODEL_PATH = OUTPUT_DIR / "mobilenetv2_civicguard.keras"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("Data root:", ACTIVE_DATA_ROOT)
print("Classes:", ACTIVE_CLASSES)

## 6. Load the dataset (80/20 train/validation split)

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    ACTIVE_DATA_ROOT,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
val_ds = keras.utils.image_dataset_from_directory(
    ACTIVE_DATA_ROOT,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
class_names = train_ds.class_names
print("class_names:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 7. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

## 8. Compile and train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

## 9. Evaluate

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print({"validation_loss": float(loss), "validation_accuracy": float(accuracy)})

## 10. Plot training curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 11. Save the model and class labels

In [ ]:
model.save(MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")

labels_path = OUTPUT_DIR / "class_names.json"
labels_path.write_text(json.dumps(class_names, indent=2), encoding="utf-8")
print(f"Saved labels to {labels_path}")

## 12. Download your trained model

Colab's disk is temporary — grab your files before the session ends.

In [ ]:
from google.colab import files

files.download(str(MODEL_PATH))
files.download(str(labels_path))

## 13. Adding `blocked_drain` and `sewage_overflow` later

Once you have images for these two classes:

**Upload a zip from your laptop:**
```python
from google.colab import files
uploaded = files.upload()  # pick blocked_drain.zip

import zipfile
with zipfile.ZipFile("blocked_drain.zip", "r") as z:
    z.extractall("data/processed/blocked_drain")
```

**Or use Google Drive** (better if you'll reuse it across sessions):
```python
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("/content/drive/MyDrive/civicguard_data/blocked_drain",
                "data/processed/blocked_drain", dirs_exist_ok=True)
```

Then just re-run from **Section 4** onward — it auto-detects the newly non-empty folders and retrains on all 5 classes without any other changes.